# Metadata filtering and access-control patterns: the real-library code of the lessons

Booster LEAP, Advanced RAG Engineering. The cells below are the library versions of the examples in the lessons (the lessons first build the same mechanism in plain Python in your browser). Each cell was run once and its output is saved below it.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use it:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again.

**What needs what:** the first run downloads the small embedding model all-MiniLM-L6-v2 (about 90 MB; it needs internet but no GPU), and chromadb is installed (about 100 MB of packages). No secret or key is used in this notebook. The install line pins the versions that were used to produce the saved outputs; Colab may resolve the dependencies of those packages differently.

In [ ]:
%pip -q install sentence-transformers==6.3.0 numpy==2.5.2 chromadb==1.5.9

## Tenant isolation: the first boundary

chromadb: dropping results after the query against passing the condition into the query.

In [ ]:
docs = {
    "acme-refund": {"text": "Acme Freight refunds a late delivery fee within 7 days of the claim.", "tenant": "acme", "allow": ["agent"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "birch-refund": {"text": "Birch Clinics refunds a missed appointment fee within 14 days if the clinic cancelled.", "tenant": "birch", "allow": ["agent"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "faq-trash": {"text": "Deleted tickets stay in the trash for 30 days and are then removed for good.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "leave-apply": {"text": "Request leave in the HR portal at least 5 working days before the first day of leave.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "run-sev": {"text": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "acme-hours": {"text": "Acme Freight support is open from 08:00 to 20:00 on working days.", "tenant": "acme", "allow": ["agent"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "acme-contract": {"text": "Acme Freight contract: 40 Business seats, renewal on 1 October, discount 12 percent.", "tenant": "acme", "allow": ["admin"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "birch-contract": {"text": "Birch Clinics contract: 12 Team seats, renewal on 1 March, discount 5 percent.", "tenant": "birch", "allow": ["admin"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "faq-uptime": {"text": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "birch-ticket-97": {"text": "Ticket 97: Lina Das (lina.das@example.com, +91 90000 00097) asks to move an appointment.", "tenant": "birch", "allow": ["agent"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
}

import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
def embed(texts):
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=False).tolist()

client = chromadb.EphemeralClient()                       # in memory, nothing is written to disk
col = client.create_collection("orbit-help", embedding_function=None, configuration={"hnsw": {"space": "cosine"}})
col.add(ids=list(docs), embeddings=embed([d["text"] for d in docs.values()]),
        metadatas=[{"tenant": d["tenant"]} for d in docs.values()])

tenant = "acme"                                           # taken from the signed-in session
for question in ("how long until a refund is paid", "how long until a missed appointment fee is refunded"):
    vector = embed([question])
    top = col.query(query_embeddings=vector, n_results=3)["ids"][0]
    after = [d for d in top if docs[d]["tenant"] in (tenant, "shared")]
    before = col.query(query_embeddings=vector, n_results=3, where={"tenant": {"$in": [tenant, "shared"]}})["ids"][0]
    print(question)
    print("  filter after :", after)
    print("  filter before:", before)

how long until a refund is paid
  filter after : ['acme-refund', 'faq-trash']
  filter before: ['acme-refund', 'faq-trash', 'acme-hours']
how long until a missed appointment fee is refunded
  filter after : ['acme-refund']
  filter before: ['acme-refund', 'faq-uptime', 'faq-trash']


## Role-based filtering: who inside the company may read

chromadb with $and, $or and $contains, then two traps with missing and empty values.

In [ ]:
docs = {
    "leave-annual": {"text": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "hr-salary": {"text": "Salary bands: grade 3 starts at 900000 rupees a year and grade 4 at 1300000 rupees.", "tenant": "orbit", "allow": ["hr"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "exp-travel": {"text": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "sec-incident": {"text": "Incident note: an access key was exposed on the build server and revoked the same day.", "tenant": "orbit", "allow": ["security"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "sec-devices": {"text": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "fin-discount": {"text": "Finance may approve a customer discount of up to 20 percent; the director decides above that.", "tenant": "orbit", "allow": ["finance"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "faq-trash": {"text": "Deleted tickets stay in the trash for 30 days and are then removed for good.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "run-sev-draft": {"text": "Draft, withdrawn: Sev 1 response within 30 minutes.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": True},
    "acme-contract": {"text": "Acme Freight contract: 40 Business seats, renewal on 1 October, discount 12 percent.", "tenant": "acme", "allow": ["admin"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
}
users = {
    "asha": {"tenant": "orbit", "groups": ["staff"]},
    "meera": {"tenant": "orbit", "groups": ["staff", "hr"]},
}

import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
def embed(texts):
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=False).tolist()

col = chromadb.EphemeralClient().create_collection("orbit-roles", embedding_function=None, configuration={"hnsw": {"space": "cosine"}})
col.add(ids=list(docs), embeddings=embed([d["text"] for d in docs.values()]),
        metadatas=[{"tenant": d["tenant"], "allow": d["allow"], "deleted": d["deleted"]} for d in docs.values()])

def where_for(user):
    groups = user["groups"] + ["everyone"]
    return {"$and": [{"tenant": {"$in": [user["tenant"], "shared"]}},
                     {"$or": [{"allow": {"$contains": g}} for g in groups]},
                     {"deleted": False}]}

vector = embed(["what is the starting salary for grade 4"])
for name, user in users.items():
    print(name, col.query(query_embeddings=vector, n_results=3, where=where_for(user))["ids"][0])

col.add(ids=["no-flag"], embeddings=embed(["a note without a deleted field"]), metadatas=[{"tenant": "orbit", "allow": ["staff"]}])
print("deleted equals False finds no-flag:", "no-flag" in col.get(where={"deleted": False})["ids"])
print("deleted not True finds no-flag   :", "no-flag" in col.get(where={"deleted": {"$ne": True}})["ids"])
try:
    col.add(ids=["empty"], embeddings=embed(["x"]), metadatas=[{"allow": []}])
except ValueError:
    print("an empty allow-list is rejected by add")

asha ['exp-travel', 'leave-annual', 'faq-trash']
meera ['hr-salary', 'exp-travel', 'leave-annual']
deleted equals False finds no-flag: False
deleted not True finds no-flag   : True
an empty allow-list is rejected by add


## Personal data and redaction

Cosine similarity of three questions with a ticket before and after redaction.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = {
    "original": 'Ticket 812: Ravi Menon (ravi.menon@example.com, +91 90000 00812) says the export link expired.',
    "email, phone removed": 'Ticket 812: Ravi Menon ([EMAIL], [PHONE]) says the export link expired.',
    "name removed too": 'Ticket 812: [NAME] ([EMAIL], [PHONE]) says the export link expired.',
    "other ticket, redacted": 'Ticket 97: Lina Das ([EMAIL], [PHONE]) asks to move an appointment.',
}
questions = ["my export link stopped working", "what did Ravi Menon report", "who uses ravi.menon@example.com"]

vectors = model.encode(list(texts.values()), normalize_embeddings=True, show_progress_bar=False)
for question in questions:
    q = model.encode(question, normalize_embeddings=True, show_progress_bar=False)
    print(question)
    for name, v in zip(texts, vectors):
        print(f"  {name:24s} {float(q @ v):.3f}")

my export link stopped working
  original                 0.595
  email, phone removed     0.576
  name removed too         0.590
  other ticket, redacted   -0.071
what did Ravi Menon report
  original                 0.269
  email, phone removed     0.310
  name removed too         0.070
  other ticket, redacted   0.106
who uses ravi.menon@example.com
  original                 0.284
  email, phone removed     0.334
  name removed too         0.185
  other ticket, redacted   0.238


## Time and version metadata

chromadb with a where filter on dates, using small hand-made vectors so the order is fixed.

In [ ]:
import math
import chromadb

# ids in the saved similarity order of "how many leave days can I carry over"
order = ["leave-carry-old", "leave-carry-next", "leave-carry", "leave-annual"]
dates = {"leave-carry-old": (20230101, 20250331), "leave-carry-next": (20270101, 99991231),
         "leave-carry": (20250401, 20261231), "leave-annual": (20250401, 99991231)}
vectors = [[math.cos(n / 10), math.sin(n / 10)] for n in range(len(order))]   # hand-made: keeps that order

col = chromadb.EphemeralClient().create_collection("rules", metadata={"hnsw:space": "cosine"})
col.add(ids=order, embeddings=vectors,
        metadatas=[{"valid_from": dates[i][0], "valid_to": dates[i][1], "deleted": False} for i in order])

def ask(where=None):
    return col.query(query_embeddings=[[1.0, 0.0]], n_results=2, where=where)["ids"][0]

print("no filter:", ask())
for day in (20240601, 20260601, 20270301):
    where = {"$and": [{"deleted": False}, {"valid_from": {"$lte": day}}, {"valid_to": {"$gte": day}}]}
    print(day, ask(where))

try:
    ask({"valid_from": {"$lte": "2026-06-01"}})
except ValueError:
    print("a date written as a string is refused by $lte")

no filter: ['leave-carry-old', 'leave-carry-next']
20240601 ['leave-carry-old']
20260601 ['leave-carry', 'leave-annual']
20270301 ['leave-carry-next', 'leave-annual']
a date written as a string is refused by $lte


## Testing for leaks

Four where filters in chromadb, each checked against the forbidden ids.

In [ ]:
import chromadb

ids = ["birch-refund", "acme-refund", "acme-contract", "acme-hours"]
meta = [{"tenant": "birch", "group": "agent"}, {"tenant": "acme", "group": "agent"},
        {"tenant": "acme", "group": "admin"}, {"tenant": "acme", "group": "agent"}]
vectors = [[1.0, 0.0], [1.0, 0.1], [1.0, 0.2], [1.0, 0.3]]        # hand-made: birch-refund is nearest
col = chromadb.EphemeralClient().create_collection("leaks", metadata={"hnsw:space": "cosine"})
col.add(ids=ids, embeddings=vectors, metadatas=meta)

mine, role = {"tenant": "acme"}, {"group": "agent"}
filters = {"correct ($and)": {"$and": [mine, role]}, "broken ($or)": {"$or": [mine, role]},
           "tenant only": mine, "no filter": None}
forbidden = {"birch-refund", "acme-contract"}                      # written by hand for acme-agent

for name, where in filters.items():
    got = col.query(query_embeddings=[[1.0, 0.0]], n_results=4, where=where)["ids"][0]
    print(f"{name:15s}", "ok" if not forbidden & set(got) else f"LEAK {sorted(forbidden & set(got))}")

correct ($and)  ok
broken ($or)    LEAK ['acme-contract', 'birch-refund']
tenant only     LEAK ['acme-contract']
no filter       LEAK ['acme-contract', 'birch-refund']


## Notebook: access control from end to end

chromadb with a where filter, checked against the saved results of all 18 cases.

In [ ]:
docs = {
    "leave-annual": {"text": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "leave-sick": {"text": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "leave-parental": {"text": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "leave-carry": {"text": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "2026-12-31", "deleted": False},
    "leave-apply": {"text": "Request leave in the HR portal at least 5 working days before the first day of leave.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "exp-meals": {"text": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "exp-approval": {"text": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "exp-travel": {"text": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "sec-passwords": {"text": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "sec-devices": {"text": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "sec-data": {"text": "Never copy customer data into chat tools, personal notes or public AI services.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "faq-seats": {"text": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "faq-sso": {"text": "Single sign-on with SAML is available on the Business plan and above.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "faq-trash": {"text": "Deleted tickets stay in the trash for 30 days and are then removed for good.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "faq-region": {"text": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "faq-uptime": {"text": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "run-sev": {"text": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "run-rollback": {"text": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "kb-err4012": {"text": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "kb-err4013": {"text": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "kb-err4021": {"text": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "kb-err5031": {"text": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "kb-err5032": {"text": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "kb-err5103": {"text": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "kb-login-loop": {"text": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "kb-slow": {"text": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "kb-mail": {"text": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "kb-merge": {"text": "Two tickets from the same customer can be merged. The older ticket keeps its number.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "acme-refund": {"text": "Acme Freight refunds a late delivery fee within 7 days of the claim.", "tenant": "acme", "allow": ["agent"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "birch-refund": {"text": "Birch Clinics refunds a missed appointment fee within 14 days if the clinic cancelled.", "tenant": "birch", "allow": ["agent"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "acme-hours": {"text": "Acme Freight support is open from 08:00 to 20:00 on working days.", "tenant": "acme", "allow": ["agent"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "birch-hours": {"text": "Birch Clinics support is open from 09:00 to 17:00 on working days.", "tenant": "birch", "allow": ["agent"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "acme-contract": {"text": "Acme Freight contract: 40 Business seats, renewal on 1 October, discount 12 percent.", "tenant": "acme", "allow": ["admin"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "birch-contract": {"text": "Birch Clinics contract: 12 Team seats, renewal on 1 March, discount 5 percent.", "tenant": "birch", "allow": ["admin"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "acme-ticket-812": {"text": "Ticket 812: Ravi Menon (ravi.menon@example.com, +91 90000 00812) says the export link expired.", "tenant": "acme", "allow": ["agent"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "birch-ticket-97": {"text": "Ticket 97: Lina Das (lina.das@example.com, +91 90000 00097) asks to move an appointment.", "tenant": "birch", "allow": ["agent"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "hr-salary": {"text": "Salary bands: grade 3 starts at 900000 rupees a year and grade 4 at 1300000 rupees.", "tenant": "orbit", "allow": ["hr"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "fin-discount": {"text": "Finance may approve a customer discount of up to 20 percent; the director decides above that.", "tenant": "orbit", "allow": ["finance"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "sec-incident": {"text": "Incident note: an access key was exposed on the build server and revoked the same day.", "tenant": "orbit", "allow": ["security"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": False},
    "leave-carry-old": {"text": "Up to 10 unused annual leave days may be carried over and must be used by 30 June.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2023-01-01", "valid_to": "2025-03-31", "deleted": False},
    "leave-carry-next": {"text": "Up to 3 unused annual leave days may be carried over and must be used by 28 February.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2027-01-01", "valid_to": "9999-12-31", "deleted": False},
    "exp-approval-old": {"text": "Claims above 3000 rupees need a receipt and manager approval.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2023-01-01", "valid_to": "2025-03-31", "deleted": False},
    "faq-trash-old": {"text": "Deleted tickets stay in the trash for 90 days and are then removed for good.", "tenant": "shared", "allow": ["everyone"], "valid_from": "2023-01-01", "valid_to": "2025-03-31", "deleted": False},
    "run-sev-draft": {"text": "Draft, withdrawn: Sev 1 response within 30 minutes.", "tenant": "orbit", "allow": ["staff"], "valid_from": "2025-04-01", "valid_to": "9999-12-31", "deleted": True},
}
users = {
    "asha": {"tenant": "orbit", "groups": ["staff"]},
    "farid": {"tenant": "orbit", "groups": ["staff", "finance"]},
    "meera": {"tenant": "orbit", "groups": ["staff", "hr"]},
    "acme-agent": {"tenant": "acme", "groups": ["agent"]},
    "acme-admin": {"tenant": "acme", "groups": ["agent", "admin"]},
    "birch-agent": {"tenant": "birch", "groups": ["agent"]},
}
cases = {
    "a01": ("acme-agent", "how long until a refund is paid",
            ["acme-refund", "faq-trash", "acme-hours"]),
    "a02": ("birch-agent", "how long until a refund is paid",
            ["birch-refund", "faq-trash", "faq-uptime"]),
    "a03": ("acme-agent", "when is support open",
            ["acme-hours", "faq-sso", "acme-ticket-812"]),
    "a04": ("birch-agent", "when is support open",
            ["birch-hours", "faq-sso", "birch-ticket-97"]),
    "a05": ("acme-admin", "when does our contract renew",
            ["acme-contract", "acme-hours", "faq-uptime"]),
    "a06": ("asha", "how many leave days can I carry over",
            ["leave-carry", "leave-annual", "leave-apply"]),
    "a07": ("asha", "which claims need a receipt",
            ["exp-approval", "faq-sso", "kb-merge"]),
    "a08": ("meera", "what is the starting salary for grade 4",
            ["hr-salary", "exp-travel", "exp-meals"]),
    "a09": ("farid", "how big a discount can finance approve",
            ["fin-discount", "faq-uptime", "exp-approval"]),
    "a10": ("asha", "can I fly business class",
            ["exp-travel", "faq-sso", "leave-carry"]),
    "a11": ("acme-agent", "my export link stopped working",
            ["kb-err5032", "acme-ticket-812", "kb-err5031"]),
    "a12": ("birch-agent", "how long do deleted tickets stay in the trash",
            ["faq-trash", "kb-merge", "kb-slow"]),
    "p01": ("asha", "what is the starting salary for grade 4",
            ["exp-travel", "exp-meals", "faq-seats"]),
    "p02": ("acme-agent", "how long until a missed appointment fee is refunded",
            ["acme-refund", "faq-uptime", "faq-trash"]),
    "p03": ("acme-agent", "what discount is in the contract",
            ["faq-uptime", "faq-sso", "faq-seats"]),
    "p04": ("asha", "was an access key exposed",
            ["sec-devices", "sec-passwords", "kb-err4013"]),
    "p05": ("birch-agent", "what did Ravi Menon report",
            ["faq-region", "birch-ticket-97", "kb-err5031"]),
    "p06": ("farid", "how many seats are in the Birch Clinics contract",
            ["faq-seats", "faq-sso", "kb-merge"]),
}

import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
ids = list(docs)
vectors = model.encode([docs[i]["text"] for i in ids], normalize_embeddings=True)

def number(day):
    return int(day.replace("-", ""))

col = chromadb.EphemeralClient().create_collection("acl", metadata={"hnsw:space": "cosine"})
col.add(ids=ids, embeddings=vectors.tolist(), metadatas=[
    {"tenant": d["tenant"], "group": d["allow"][0], "deleted": d["deleted"],
     "valid_from": number(d["valid_from"]), "valid_to": number(d["valid_to"])} for d in docs.values()])

same = 0
for case, (name, question, expected) in cases.items():
    user, today = users[name], number("2026-06-01")
    where = {"$and": [{"tenant": {"$in": [user["tenant"], "shared"]}},
                      {"group": {"$in": user["groups"] + ["everyone"]}},
                      {"deleted": False},
                      {"valid_from": {"$lte": today}}, {"valid_to": {"$gte": today}}]}
    q = model.encode([question], normalize_embeddings=True)
    got = col.query(query_embeddings=q.tolist(), n_results=3, where=where)["ids"][0]
    assert got == expected, (case, got, expected)
    same += 1
print("cases with the same three ids as the saved ranking:", same, "of", len(cases))

cases with the same three ids as the saved ranking: 18 of 18
